<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 1 - Analyse exploratoire (part2) - Nettoyage et aggregation séquentiel</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [1]:
# Roots
import gc
import numpy as np
import pandas as pd
# import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Stats
from scipy.stats import zscore, chi2_contingency, f_oneway, chi2

In [2]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [3]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [4]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))


In [ ]:
# Fonctions personnelles
from notebooks.datas_manipulation.quick_clean_datas import (
    remove_duplicates,drop_empty_columns, clean_infinites
)
from notebooks.datas_manipulation.datas_assembler import merging_data
from notebooks.datas_manipulation.memory_optimizer import optimize_dtypes
from notebooks.datas_manipulation.export_datas import export_datas

from notebooks.utils.feature_aggregator import agg_features, agg_columns

In [6]:
# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

# choix de sauvegarde pour la data Xy
yes_choice = {'YES','yes','y','Y'}
save_datas = "n"

# Choix de sauvegarde pour les graphes
save_graphs = False

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [7]:
# Chemin du dossier de données brut
datas_path = root_path /'datas'/'raw_datas'/'Projet+Mise+en+prod+-+home-credit-default-risk'

<span style="color:blue;font-weight:bold"> Rappel des features </span>

In [ ]:
# fichier de description
df_description = pd.read_csv(datas_path/"HomeCredit_columns_description.csv",encoding='latin1')
df_description[["Row","Description"]].T

,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
...,...,...,...,...,...
214,217,installments_payments.csv,NUM_INSTALMENT_NUMBER,On which installment we observe payment,NaN
215,218,installments_payments.csv,DAYS_INSTALMENT,When the installment of previous credit was su...,time only relative to the application
216,219,installments_payments.csv,DAYS_ENTRY_PAYMENT,When was the installments of previous credit p...,time only relative to the application
217,220,installments_payments.csv,AMT_INSTALMENT,What was the prescribed installment amount of ...,NaN


In [ ]:
print(f"nombre de features uniques:{df_description["Row"].nunique()}")
# on regroupe suivant Row est on créée une liste des tables associées. On obtient
# une Series avec les Row en index et la liste des tables en valeurs
duplicated_features = df_description.groupby('Row')['Table'].apply(list).reset_index(drop=True)
# on ne garde que les features dupliquées
duplicated_features = duplicated_features[duplicated_features['Table'].map(len) > 1]

final_view = duplicated_features.set_index('Row').T
display(final_view)

nombre de features uniques:196
['SK_ID_CURR', 'AMT_ANNUITY', 'SK_BUREAU_ID', 'MONTHS_BALANCE', 'SK_ID_PREV ', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF', 'NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'NAME_TYPE_SUITE']


,Unnamed: 0,Table,Row,Description,Special
0,125,bureau.csv,SK_ID_CURR,ID of loan in our sample - one loan in our sam...,hashed
1,141,bureau.csv,AMT_ANNUITY,Annuity of the Credit Bureau credit,NaN
2,142,bureau_balance.csv,SK_BUREAU_ID,Recoded ID of Credit Bureau credit (unique cod...,hashed
3,146,POS_CASH_balance.csv,SK_ID_CURR,ID of loan in our sample,NaN
4,147,POS_CASH_balance.csv,MONTHS_BALANCE,Month of balance relative to application date ...,time only relative to the application
5,153,credit_card_balance.csv,SK_ID_PREV,ID of previous credit in Home credit related t...,hashed
6,154,credit_card_balance.csv,SK_ID_CURR,ID of loan in our sample,hashed
7,155,credit_card_balance.csv,MONTHS_BALANCE,Month of balance relative to application date ...,time only relative to the application
8,173,credit_card_balance.csv,NAME_CONTRACT_STATUS,"Contract status (active signed,...) on the pre...",NaN
9,174,credit_card_balance.csv,SK_DPD,DPD (Days past due) during the month on the pr...,NaN


<span style="color:red">Remarque: erreur dans le nommage SK_BUREAU_ID dans le fichier de description ==> dans la data c'est bien SK_ID_BUREA</span>

On retrouve une description des fichiers ainsi qu'un diagramme de relation sur https://www.kaggle.com/c/home-credit-default-risk/data.


**Rappel des fichiers**

En comptant le fichier descriptif, on a 10 fichiers (c'est plus que le diagramme car train|test ensemble tandis que le fichier *HomeCredit_columns_description* et *sample_submission* non plus car l'un décrit les features et l'autre). Les fichiers sont reliés par le SK_ID_CURR (l'ID client CHEZ HOME CREDIT) et une liaison complémentaire se fait avec les autres ID (SK_ID_BUREAU pour bureau et bureau_balance / SK_ID_PREV pour previous_application avec POS_CASH_balance, installments_payments et credit_card_balance). Pour ce qui est de leur contenu:
- **application_train/test**: Les données principales (Démographie, revenus, montant du prêt). Une ligne = Un prêt.
- **bureau**: Données de tous les emprunts enregistrés au Bureau du Crédit (toutes les institutions incluant aussi Home Credit).
- **bureau_balance**: Historique mensuel des crédits (état de remboursement).
- **previous_application**: Toutes les demandes de prêts faites par le client chez Home Credit par le passé.
- **POS_CASH_balance**: Historique mensuel des soldes des anciens prêts (Point of Sale et Cash).
- **installments_payments**: Historique de chaque paiement réalisé par rapport aux anciens prêts (réussi ou echoué).
- **credit_card_balance**: Historique mensuel de l'utilisation des cartes de crédit du client.


<span style="color:black;font-size:1em;background-color:yellow"> CREDIT BUREAU EST UNE INSTITUTION QUI RECENSE LES PRETS CONTRAIREMENT A HOME CREDIT QUI EST LA BANQUE PRETEUSE ICI!!!</span>

<span style="color:blue;font-weight:bold"> Dataframe prinipale: df_train_test</span>

In [10]:
miss = 0.8
miss_percent = int(miss*100)

In [12]:
df_train_test = pd.read_parquet(datas_path/f"train_test{miss_percent}_cleaned.parquet")

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de la branche bureau</span>

Comme il a été dit précédemment au vu du nombre de lignes qui dépasse très largement le nombre d'observations train/test, en partant du postulat que ce dépassement (de plusieurs millions tout de même pour certains fichiers) est majoritairement lié au fait que plusieurs lignes concernent un même client (avec éventuellement un peu de doublons), on va devoir regarder séparemment leur contenu et aggreger afin de n'avoir qu'une observation par client.

- **bureau** contient l'ensemble des prêts déclarés dans une institution des clients. Chaque ligne correspond a un crédit et caractérisée par deux ID:
    - SK_ID_CURR: l'ID du client chez Home Credit
    - SK_ID_BUREAU: L'ID d'un prêt enregistré chez Credit Bureau
- **bureau_balance** contient l'historique mensuel de tous les prêts chez Credit Bureau (CrB). Chaque ligne correspond à l'état mensuel d'un prêt au sein de CrB caractérisée par:
    - SK_ID_BUREAU
    - MONTHS_BALANCE: $\leq 0$ avec 0, le mois courant
    - STATUS: Etat du remboursement à ce jour [0-5,C,X], 0 a 5 pour un retard sur le remboursemnt (plus la valeur est elevée plus il y a du retard avec 0 pas de retard et 5 pour prêt revendu), C pour un prêt cloturé et X pour dire statut inconnu. 



In [ ]:
steps = ["bureau", "prev_app", "pos_cash", "credit_card", "installements"]
ID_list = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

In [ ]:
temp_file_path_bureau = datas_path/steps[0]/f"df_train_test_{steps[0]}.parquet"

if temp_file_path_bureau.exists():
    df_train_test = pd.read_parquet(temp_file_path_bureau)
else:
    # On importe le premier fichier sur lequel on aggrege la donnée d'abord puis 
    # qui sera fusionné dans le second (et garbage collecté via la fonction d'agg).

    # Bureau Balance
    df_bureau_balance = pd.read_parquet(datas_path/f'bureau_balance{miss_percent}_cleaned.parquet')
    # Agrégation de Bureau Balance
    df_bureau_balance_agg = agg_features(
        df_bureau_balance, 'SK_ID_BUREAU', 'BB', drop_columns=ID_list
    )

    # Bureau (On joint BB dedans d'abord)
    df_bureau = pd.read_parquet(datas_path / f"bureau{miss_percent}_cleaned.parquet")
    # On attache les infos de balance à bureau puis on le garbage collect
    df_bureau = merging_data(df_bureau, df_bureau_balance_agg, on='SK_ID_BUREAU', how='left')


    # Agrégation finale pour n'avoir qu'une ligne par client
    df_bureau_agg = agg_features(df_bureau, 'SK_ID_CURR', 'BUREAU', drop_columns=ID_list)


    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, df_bureau_agg, on='SK_ID_CURR', how='left')
    
    # on sauvegarde temporairement le df_bureau nettoyé
    export_datas(df_train_test, datas_path, step = steps[0], prefix = "df_train_test_")

In [16]:
df_train_test.head(10)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_CREDIT_TYPE_Interbank credit,BUREAU_CREDIT_TYPE_Loan for business development,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending),BUREAU_CREDIT_TYPE_Loan for the purchase of equipment,BUREAU_CREDIT_TYPE_Loan for working capital replenishment,BUREAU_CREDIT_TYPE_Microloan,BUREAU_CREDIT_TYPE_Mobile operator loan,BUREAU_CREDIT_TYPE_Mortgage,BUREAU_CREDIT_TYPE_Real estate loan,BUREAU_CREDIT_TYPE_Unknown type of loan
0,100002.0,1.0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100003.0,0.0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004.0,0.0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006.0,0.0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007.0,0.0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008.0,0.0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,100009.0,0.0,Cash loans,F,Y,Y,1,171000.0,1560726.0,41301.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,100010.0,0.0,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,...,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011.0,0.0,Cash loans,F,N,Y,0,112500.0,1019610.0,33826.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,100012.0,0.0,Revolving loans,M,N,Y,0,135000.0,405000.0,20250.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df_train_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 218 entries, SK_ID_CURR to BUREAU_CREDIT_TYPE_Unknown type of loan
dtypes: float32(71), float64(92), int16(2), int8(36), object(17)
memory usage: 406.3+ MB


On peut voir qu'on a pratiquement doubler le nombre de features (mais on s'est évité les 27 millions de ligne de bureau_balance). On avait pu prendre connaissance précédemment des features doublons (donc il faut attendre la fusion pour manipuler) ainsi que la description les concernant.
- on peut donc réaliser des features engineering
- on peut aggréger des colonnes ensemble
- supprimer des colonnes vides (80 ou 90%?)

In [ ]:
# on peut suppr l'ID BUREAU
df_train_test.drop(columns='SK_ID_BUREAU',inplace=True,errors='ignore')

<span style="color:blue;font-weight:bold"> Aggregation des FLAG </span>

In [ ]:
# Possession (car & realty) ==> un proprietaire réduit le risque de défaut in fine
df_train_test = agg_columns(df_train_test, 'POSSESSION', ["FLAG_OWN_CAR","FLAG_OWN_REALTY"])

In [ ]:
# Documentation (2 a 21) ==> Nombre de doc admin rempli par le client, moins de risque si plus
flag_doc_list = [col for col in df_train_test.columns if col.startswith('FLAG_DOCUMENT')]
df_train_test=agg_columns(df_train_test,'FILLED_DOC_COUNT',flag_doc_list)

In [ ]:
# EXT_SOURCE (1,2 et 3)
df_train_test=agg_columns(
    df_train_test,'EXT_SOURCE_COUNT',['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'])

In [ ]:
# Moyn de communication fourni (mail, tel, mobile..)
contact_ways_list=[
    'FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE','FLAG_CONT_MOBIL',
    'FLAG_PHONE','FLAG_EMAIL'
]
df_train_test=agg_columns(df_train_test,'CONTACT_WAYS_COUNT',contact_ways_list)

In [ ]:
# Informations erronees qui rend le client suspect
mismatch_info_list=[
    'REG_REGION_NOT_LIVE_REGION',
    'REG_CITY_NOT_LIVE_CITY',
    'REG_CITY_NOT_WORK_CITY'
]
df_train_test=agg_columns(df_train_test,'MISMATCH_INFO_COUNT', mismatch_info_list)

<span style="color:blue;font-weight:bold"> Feature engineering  </span>

In [ ]:
# Taux d'endettement
df_train_test['DEBT_RATIO'] = \
    df_train_test['AMT_ANNUITY'] / df_train_test['AMT_INCOME_TOTAL']

df_train_test['PAYMENT_RATE'] = \
    df_train_test['AMT_ANNUITY'] / df_train_test['AMT_CREDIT']

df_train_test['PURCHASING_POWER_RATIO'] = \
    df_train_test['AMT_INCOME_TOTAL'] / df_train_test['AMT_CREDIT']

df_train_test['DAYS_EMPLOYED_RATIO'] = \
    df_train_test['DAYS_EMPLOYED'] / df_train_test['DAYS_BIRTH']

In [ ]:
# impossible pour le moment de faire des chi2, anova , heatmap etc... donc corr simple
# on va suppr les features très proche de zero
correlations = df_train_test.corr()['TARGET'].sort_values()

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de la branche previous_application</span>

A la différence de la branche bureau pour laquelle la sous-branche, bureau_balance n'est liée qu'à bureau (via SK_ID_BUREAU), les fichiers previous_application, POS_CASH_balance, installments_payments et credit_card_balance sont tous liés directement à train_test c'est pourquoi on peut se permettre de tous traiter et merger directement vers train_test. De plus, moins on introduit d'étapes intermédiaire modifiant la donnée moins il y aura de biais. Et pour finir le point central reste la gestion de mémoire, installments_payments et credit_card_balance sont les fichiers les plus lourds, ainsi aggregation et jointure seront particulièrement gourmandes et pour minimiser le coût, il sera préférable de procéder ainsi.

In [ ]:
temp_file_path_prev_app = datas_path/steps[1]/f"df_train_test_{steps[1]}.parquet"

if temp_file_path_prev_app.exists():
    df_train_test = pd.read_parquet(temp_file_path_prev_app)
else:
    prev_app = pd.read_parquet(datas_path/"previous_application{miss_percent}_cleaned.parquet")
    
    # # Feature Engineering spécifique (Avant agrégation) a faire mainteannt?
    # prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    
    # aggregation de previous application
    prev_agg = agg_features(prev_app, 'SK_ID_CURR', 'PREV', drop_columns=ID_list)
    
    # Fusion avec le df_test_train
    df_train_test = merging_data(df_train_test, prev_agg, on='SK_ID_CURR', how='left')
    
    # on sauvegarde temporairement le df_previous_application nettoyé
    export_datas(df_train_test, datas_path, step = steps[1], prefix = "df_train_test_")

In [ ]:
df_train_test.head(10)